# Appendix A — Flue-Gas Flow-Rate Derivation and Validation (Eq. 5.1)

**Fulfills the promise made in the thesis at §5.1.2 (p. 56) and §5.10.1 (p. 82):**

> "A full derivation and verification against Kim and Léonard's (2025) own published data points,
> reproducing their Nwaoha et al. (2018) validation case TEC to within 1.1%, is provided in the Appendix."

> "The flue-gas flow-rate derivation (Eq. 5.1) reproduces Kim and Léonard's (2025) Nwaoha et al. (2018)
> validation case TEC of €42.07 million to within 1.1%."

This notebook is self-contained and reproduces the exact functions used in `CCS_TEA_Model.ipynb`
(Cells 7 and 9) to verify that this thesis's implementation of Eq. 5.1 and the Kim & Léonard (2025)
TEC correlation (Eq. 5.3) is correct, independent of the published source's own internal validation.

## A.1 Derivation of Eq. 5.1

The Kim and Léonard (2025) CAPEX and specific-duty correlations (Eqs. 5.3, 5.8, 5.9) take flue gas
flow rate, *F* [10³ Nm³/h], and CO₂ inlet mole fraction, *x_CO2*, as direct inputs, rather than annual
captured tonnage. Converting from tonnage to flow rate requires the CO₂ density at standard conditions
(0 °C, 1 atm):

$$\rho_{CO2} = \frac{M_{CO2}}{V_{molar}} = \frac{44.01 \text{ g/mol}}{22.414 \text{ L/mol}} = 1.9635\ \text{kg/Nm}^3$$

The annual mass of CO₂ actually captured, $Q_{captured}$ [t/yr], is delivered over $t_{op}$ operating
hours per year at capture rate $\eta_{cap}$. The corresponding flue-gas volumetric flow rate is:

$$F = \frac{Q_{captured} \times 1000}{t_{op} \times x_{CO2} \times \rho_{CO2} \times \eta_{cap} \times 1000}\quad [10^3\ \text{Nm}^3/\text{h}] \qquad \text{(Eq. 5.1)}$$

where $x_{CO2}$ is a decimal fraction (0.10 for 10 mol%), $t_{op}$ is annual operating hours, and
$\eta_{cap}$ is the capture rate (0.90, fixed per §5.1.1).

In [1]:
# ── Verbatim from CCS_TEA_Model.ipynb, Cell 7 ──────────────────────────────
M_CO2   = 44.01    # g/mol
V_molar = 22.414   # L/mol at STP
rho_CO2 = M_CO2 / V_molar   # = 1.9635 kg/Nm3

def flue_gas_flow_F(Q_captured_t_yr, x_CO2_frac, t_op_h_yr, cap_rate=0.90):
    """Eq. 5.1: F [10^3 Nm3/h] from annual captured volume.
    Q_captured: t CO2/yr actually captured (not design)
    x_CO2_frac: mole fraction (decimal)
    """
    return (Q_captured_t_yr * 1000) / (
        t_op_h_yr * x_CO2_frac * rho_CO2 * cap_rate * 1000)

print(f"CO2 density at STP: {rho_CO2:.4f} kg/Nm3")

CO2 density at STP: 1.9635 kg/Nm3


## A.2 The Kim & Léonard (2025) TEC correlation (Eq. 5.3)

$$TEC\ [\text{M}€_{2023}] = \alpha + (\beta \cdot x_{CO2}^{\,n} + \gamma)\cdot F^{m}$$

with fitted coefficients $\alpha=2.1673$, $\beta=0.8092$, $\gamma=-0.00332$, $n=0.5291$, $m=0.8391$
(Kim & Léonard, 2025, Table 14).

In [2]:
# ── Verbatim from CCS_TEA_Model.ipynb, Cell 9 ──────────────────────────────
kl = dict(alpha=2.1673, beta=0.8092, gamma=-0.00332, n=0.5291, m=0.8391)

def kim_leonard_TEC_MEur2023(x_CO2_frac, F_1000Nm3h):
    """Eq. 5.3: TEC [M EUR 2023]. x_CO2_frac as decimal (0.10 for 10 mol%)."""
    return (kl["alpha"]
            + (kl["beta"] * x_CO2_frac**kl["n"] + kl["gamma"])
            * F_1000Nm3h**kl["m"])

## A.3 Validation against Kim and Léonard's (2025) Nwaoha et al. (2018) benchmark case

Kim and Léonard (2025) validate their own correlations against the cement-plant case study reported
in Nwaoha et al. (2018), reproducing that paper's simulation-based TEC to within 1.15% (their Table D1,
their Fig. 7). Because Eq. 5.1 and the Kim & Léonard TEC correlation (Eq. 5.3) are both implemented
independently in this thesis's model, the same benchmark case provides a direct check that *this
thesis's implementation* — not just the published correlation — reproduces the literature figure.

**Benchmark case inputs (Kim & Léonard 2025, Table 10 — process parameters from Nwaoha et al. 2018):**

| Parameter | Value |
|---|---|
| Capture scale, Q_captured | 700,000 t CO₂/yr (0.7 Mt/y) |
| CO₂ inlet concentration, x_CO2 | 11.5 mol% |
| Operating hours, t_op | 8,215 h/yr |
| Capture rate, η_cap | 90% |

**Published benchmark output (Kim & Léonard 2025, Table D1):** TEC = €42.07 million (2023).

In [3]:
# ── Nwaoha (2018) benchmark case, as tabulated in Kim & Léonard (2025) ─────
Q_captured_Nwaoha = 700_000    # t/yr (0.7 Mt/y)
x_CO2_Nwaoha      = 0.115      # 11.5 mol%
t_op_Nwaoha       = 8_215      # h/yr
cap_rate_Nwaoha   = 0.90

F_Nwaoha = flue_gas_flow_F(Q_captured_Nwaoha, x_CO2_Nwaoha, t_op_Nwaoha, cap_rate_Nwaoha)
print(f"F (calculated via Eq. 5.1): {F_Nwaoha:.3f} x 10^3 Nm3/h")

TEC_calc = kim_leonard_TEC_MEur2023(x_CO2_Nwaoha, F_Nwaoha)
TEC_published = 42.07   # M EUR 2023, Kim & Leonard (2025) Table D1

pct_diff = (TEC_calc - TEC_published) / TEC_published * 100

print(f"\nTEC (this thesis's implementation): {TEC_calc:.3f} M EUR 2023")
print(f"TEC (Kim & Leonard 2025, Table D1, published): {TEC_published:.2f} M EUR 2023")
print(f"Percent difference: {pct_diff:.3f}%")
assert abs(pct_diff) < 1.2, "Validation exceeds the 1.1% figure reported in the thesis (§5.1.2, §5.10.1)"
print("\nPASS: within the 1.1% tolerance reported in the thesis (Section 5.1.2, p. 56; Section 5.10.1, p. 82).")

F (calculated via Eq. 5.1): 419.293 x 10^3 Nm3/h

TEC (this thesis's implementation): 42.531 M EUR 2023
TEC (Kim & Leonard 2025, Table D1, published): 42.07 M EUR 2023
Percent difference: 1.096%

PASS: within the 1.1% tolerance reported in the thesis (Section 5.1.2, p. 56; Section 5.10.1, p. 82).


## A.4 Interpretation

This is consistent with Kim and Léonard's own reported 1.15% difference against Nwaoha et al. (2018),
confirming that both the flue-gas flow-rate conversion (Eq. 5.1) and the TEC correlation (Eq. 5.3) are
correctly implemented in this thesis's model, independent of the published source's own validation
exercise. This closes the derivation-and-verification promise made at §5.1.2 and §5.10.1 of the thesis.

**Source of record:** the functions reproduced here are copied verbatim from `CCS_TEA_Model.ipynb`
(Cells 7 and 9) in this repository, so this validation exercises the same code used to produce the
thesis's headline LCOC results — not a re-implementation.